<a href="https://colab.research.google.com/github/padz24/Portofolio/blob/main/retrain_svm_face_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrain SVM Face Model (HOG + SVM) — Google Colab

Pipeline: **upload dataset → deteksi & alignment wajah → augmentasi → ekstraksi HOG → grid search → hard negative mining → simpan `.pkl`** dalam format yang sama dengan `svm_face_final.pkl` (class `SKLearnFaceDetector` dari modul `model_cnn`), jadi tinggal dipakai langsung di notebook webcam sebelumnya.

**Struktur dataset yang diharapkan (dalam satu file `dataset.zip`):**
```
dataset.zip
├── positive/   <- foto wajah kamu (target), banyak variasi sudut/cahaya/ekspresi
└── negative/   <- foto wajah orang lain / bukan wajah, sebanyak & sebervariasi mungkin
```
Jalankan cell 1 → 9 berurutan. Cell 7 (grid search) & 8 (hard negative mining) opsional tapi sangat disarankan.


## Cell 1 — Install dependencies

In [1]:
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
%pip install --no-cache-dir --force-reinstall "opencv-python-headless==4.11.0.86" scikit-image -q
print("Selesai. Jika diminta restart runtime, restart dulu sebelum lanjut ke Cell 2 (jangan run Cell 1 lagi).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 171.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.0/318.0 kB 252.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 192.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 241.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 266.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires

## Cell 2 — Verifikasi OpenCV

In [1]:
import cv2, os
print("cv2 version:", cv2.__version__)
print("CascadeClassifier tersedia:", hasattr(cv2, 'CascadeClassifier'))

cv2 version: 4.11.0
CascadeClassifier tersedia: True


Cell 2.1 — Generate dataset

In [ ]:
from IPython.display import display, HTML
from google.colab.output import register_callback
from base64 import b64decode
import cv2, numpy as np, os

os.makedirs('dataset/positive', exist_ok=True)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
counter = {'n': 0}

def save_frame(data_url):
    header, encoded = data_url.split(',', 1)
    arr = np.frombuffer(b64decode(encoded), dtype=np.uint8)
    frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(80, 80))
    saved = 0
    for (x, y, w, h) in faces:
        crop = frame[y:y+h, x:x+w]
        cv2.imwrite(f"dataset/positive/pos_{counter['n']:04d}.jpg", crop)
        counter['n'] += 1
        saved += 1
    return saved

register_callback('notebook.save_frame', save_frame)

display(HTML('''
<video id="video" autoplay playsinline style="width:100%;max-width:400px"></video>
<div id="status" style="color:#0f0;font:14px monospace"></div>
<script>
(async () => {
  const video = document.getElementById('video');
  const status = document.getElementById('status');
  video.srcObject = await navigator.mediaDevices.getUserMedia({video:true});
  await video.play();
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth; canvas.height = video.videoHeight;
  const ctx = canvas.getContext('2d');
  let total = 0;
  async function capture() {
    ctx.drawImage(video, 0, 0);
    const dataUrl = canvas.toDataURL('image/jpeg', 0.85);
    const res = await google.colab.kernel.invokeFunction('notebook.save_frame', [dataUrl], {});
    total += res.data['application/json'];
    status.textContent = "Tersimpan: " + total + " foto — gerakkan kepala pelan-pelan (miring, tengok, dekat-jauh)";
  }
  setInterval(capture, 800);  // ambil foto tiap 0.8 detik
})();
'''))

In [ ]:
from sklearn.datasets import fetch_lfw_people
import cv2, os

os.makedirs('dataset/negative', exist_ok=True)

lfw = fetch_lfw_people(min_faces_per_person=1, resize=1.0, color=True)
print("Total foto LFW:", len(lfw.images))

# ambil subset acak (misal 500) biar tidak kebanyakan & tidak makan waktu lama
import numpy as np
idx = np.random.choice(len(lfw.images), size=min(500, len(lfw.images)), replace=False)

for i, ix in enumerate(idx):
    img = (lfw.images[ix] * 255).astype('uint8')  # RGB float -> uint8
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(f"dataset/negative/neg_{i:04d}.jpg", img_bgr)

print("Selesai, dataset negatif:", len(os.listdir('dataset/negative')), "foto")

In [ ]:
import shutil
shutil.make_archive('dataset', 'zip', '.', 'dataset')
from google.colab import files
files.download('dataset.zip')

## Cell 3 — Upload & extract dataset.zip

In [ ]:
import zipfile, shutil
from google.colab import files

DATASET_DIR = 'dataset'

if not os.path.isdir(DATASET_DIR):
    print("Upload dataset.zip (harus berisi folder 'positive/' dan 'negative/')")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall(DATASET_DIR)

# cari folder positive/negative di dalam DATASET_DIR (jaga-jaga kalau ke-nest)
def find_dir(root, name):
    for dirpath, dirnames, _ in os.walk(root):
        if name in dirnames:
            return os.path.join(dirpath, name)
    return None

POS_DIR = find_dir(DATASET_DIR, 'positive')
NEG_DIR = find_dir(DATASET_DIR, 'negative')
assert POS_DIR and NEG_DIR, "Folder 'positive' atau 'negative' tidak ditemukan di dalam zip!"

print("positive:", POS_DIR, "->", len(os.listdir(POS_DIR)), "file")
print("negative:", NEG_DIR, "->", len(os.listdir(NEG_DIR)), "file")


## Cell 4 — Deteksi wajah + alignment (rotate berdasar posisi mata) + crop 64x64

In [ ]:
import numpy as np

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
IMG_SIZE = (64, 64)

def align_face(gray_face):
    """Rotate wajah supaya kedua mata sejajar horizontal, kalau mata terdeteksi."""
    eyes = eye_cascade.detectMultiScale(gray_face, scaleFactor=1.1, minNeighbors=5)
    if len(eyes) < 2:
        return gray_face
    eyes = sorted(eyes, key=lambda e: e[0])[:2]
    (x1, y1, w1, h1), (x2, y2, w2, h2) = eyes
    c1 = (x1 + w1 / 2, y1 + h1 / 2)
    c2 = (x2 + w2 / 2, y2 + h2 / 2)
    dy, dx = c2[1] - c1[1], c2[0] - c1[0]
    angle = np.degrees(np.arctan2(dy, dx))
    center = (gray_face.shape[1] // 2, gray_face.shape[0] // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(gray_face, M, (gray_face.shape[1], gray_face.shape[0]))

def detect_and_crop(img_path):
    """Baca gambar, deteksi wajah terbesar, align, resize ke 64x64 grayscale. Return None kalau tidak ada wajah."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(50, 50))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])  # wajah terbesar
    face = gray[y:y+h, x:x+w]
    face = align_face(face)
    return cv2.resize(face, IMG_SIZE)

print("Fungsi deteksi & alignment siap.")


## Cell 5 — Augmentasi data

In [ ]:
def augment(face_gray):
    """Return list of augmented versions (termasuk originalnya) dari satu wajah 64x64 grayscale."""
    variants = [face_gray]
    variants.append(cv2.flip(face_gray, 1))  # mirror horizontal

    h, w = face_gray.shape
    for angle in (-10, 10):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        variants.append(cv2.warpAffine(face_gray, M, (w, h)))

    variants.append(cv2.convertScaleAbs(face_gray, alpha=1.25, beta=15))   # lebih terang
    variants.append(cv2.convertScaleAbs(face_gray, alpha=0.75, beta=-15))  # lebih gelap
    return variants

print("Fungsi augmentasi siap. Tiap wajah asli akan menghasilkan", len(augment(np.zeros(IMG_SIZE, dtype=np.uint8))), "varian.")


## Cell 6 — Ekstraksi fitur HOG dari seluruh dataset (+ augmentasi)

In [ ]:
from skimage.feature import hog

def extract_hog(face_gray):
    return hog(face_gray, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm='L2-Hys')

def build_features(folder, label, use_augment=True):
    X, y, skipped = [], [], 0
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        face = detect_and_crop(path)
        if face is None:
            skipped += 1
            continue
        variants = augment(face) if use_augment else [face]
        for v in variants:
            X.append(extract_hog(v))
            y.append(label)
    print(f"{folder}: {len(y)} fitur dihasilkan, {skipped} file dilewati (wajah tidak terdeteksi)")
    return X, y

Xp, yp = build_features(POS_DIR, 1, use_augment=True)
Xn, yn = build_features(NEG_DIR, 0, use_augment=True)

X = np.array(Xp + Xn)
y = np.array(yp + yn)
print("Total dataset:", X.shape, "| positif:", sum(y == 1), "| negatif:", sum(y == 0))


## Cell 7 — Baseline train + Grid Search hyperparameter

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 0.01, 0.001],
}

grid = GridSearchCV(
    SVC(probability=True, class_weight='balanced'),
    param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV f1-score:", grid.best_score_)

model = grid.best_estimator_
print(classification_report(y_test, model.predict(X_test), target_names=['negative', 'positive']))


## Cell 8 — Hard Negative Mining (opsional, sangat disarankan)
Upload foto/frame TAMBAHAN yang berisi wajah **selain target** (misalnya kumpulan foto random dari internet, atau screenshot orang lain). Model saat ini dijalankan ke semua wajah yang terdeteksi di situ; wajah yang salah diprediksi sebagai "Match" (false positive) otomatis ditambahkan sebagai data negatif baru, lalu model dilatih ulang. Ini biasanya paling efektif menekan false-positive di dunia nyata.


In [ ]:
print("Upload beberapa foto/gambar berisi wajah ORANG LAIN (bukan target) untuk hard negative mining.")
print("Bisa skip cell ini (Runtime > tidak menjalankan) kalau belum punya data tambahan.")

hard_neg_uploaded = files.upload()

HARD_NEG_DIR = 'hard_negative_raw'
os.makedirs(HARD_NEG_DIR, exist_ok=True)
for fname, content in hard_neg_uploaded.items():
    with open(os.path.join(HARD_NEG_DIR, fname), 'wb') as f:
        f.write(content)

false_positive_faces = []
for fname in os.listdir(HARD_NEG_DIR):
    face = detect_and_crop(os.path.join(HARD_NEG_DIR, fname))
    if face is None:
        continue
    feat = extract_hog(face).reshape(1, -1)
    pred = model.predict(feat)[0]
    if pred == 1:  # model salah bilang ini "Match" padahal bukan target -> hard negative
        false_positive_faces.append(face)

print(f"Ditemukan {len(false_positive_faces)} hard negative (false positive) dari {len(os.listdir(HARD_NEG_DIR))} file.")

if false_positive_faces:
    X_hard, y_hard = [], []
    for face in false_positive_faces:
        for v in augment(face):
            X_hard.append(extract_hog(v))
            y_hard.append(0)

    X_train = np.vstack([X_train, np.array(X_hard)])
    y_train = np.concatenate([y_train, np.array(y_hard)])
    print("Data training baru setelah hard negative mining:", X_train.shape)

    # retrain dengan best params dari grid search + data tambahan
    model = SVC(probability=True, class_weight='balanced', **grid.best_params_)
    model.fit(X_train, y_train)
    print(classification_report(y_test, model.predict(X_test), target_names=['negative', 'positive']))
else:
    print("Tidak ada false positive baru — model sudah cukup baik terhadap data ini, lanjut tanpa retrain tambahan.")


## Cell 9 — Simpan model final (format sama dengan svm_face_final.pkl)

In [ ]:
import pickle

class SKLearnFaceDetector:
    def __init__(self, hog_fn=None, svm_model=None):
        self.hog = hog_fn
        self.model = svm_model
        self.trained = True

detector_new = SKLearnFaceDetector(hog, model)

OUT_NAME = 'svm_face_final_v2.pkl'
with open(OUT_NAME, 'wb') as f:
    pickle.dump(detector_new, f)

print(f"Model tersimpan sebagai {OUT_NAME}")
print("classes:", model.classes_, "| n_features_in_:", model.n_features_in_, "| kernel:", model.kernel)

files.download(OUT_NAME)
